# Monitor: from alarm to traffic light
**Internet of Production – AI, week 3, block 6** · about 45 min · work in pairs

**Thread:** The service answers a question: *does this window of energy data look like the run-up to a blockage?* A technician does not want a stream of single alarms but a **state**: green, yellow, red. In this notebook you design the traffic light, run it on a replay of the print farm, and look at what it hands over to the agents of the next block.

**Architecture**
```
replay_server.py  --Energy, State (every minute)-->  Monitor  --HTTP POST /predict-->  blockage_service.py
 (OPC UA, data of 2026)                                 |
        ^                                               v  health score -> traffic light
        +--- Maintenance/Jobs/M01_RepairNeeded = True ---+  + orders concerned (Orders/Onnnn)
```
The replay server plays the data of 2026 as if it were live: same nodes as the factory server of the agent block (`Factory/Machines/M01 ...`, `Maintenance/Jobs`, `Orders`), plus the node `Energy` (Wh in the last minute). **Why a replay?** The data are those the farm would deliver, but reproducible and fast: a blockage that takes hours can be shown in a minute.

*Prerequisite:* notebook 02 has been run (the registered models must exist). Stop the service of notebook 03 or leave it: this notebook uses the same port and the same models.

In [ ]:
import sys, subprocess, time, json, asyncio, logging, warnings
sys.path.insert(0, '..')                                           # blockage_core.py and monitor_core.py lie in the folder above
import numpy as np, pandas as pd, matplotlib.pyplot as plt, requests
from pathlib import Path
import nest_asyncio; nest_asyncio.apply()                          # lets us use await in a notebook cell
import mlflow, mlflow.pyfunc
from IPython.display import HTML, display
logging.getLogger('mlflow').setLevel(logging.ERROR); logging.getLogger('asyncua').setLevel(logging.WARNING); warnings.filterwarnings('ignore')
from blockage_core import *
from monitor_core import Monitor, announced, offline_evaluation, COLORS

DATA_NEW = Path('../../00_Data/printfarm_2026')
STORE = Path('..').resolve()
TRACKING_URI = f"sqlite:///{(STORE / 'mlflow.db').as_posix()}"
mlflow.set_tracking_uri(TRACKING_URI)
START, HOURS, SPEED = '2026-01-28 12:00', 12, 600                  # replay: 12 hours of 28 Jan 2026, 600 simulated minutes per real minute
SERVICE_URL = 'http://127.0.0.1:5005'
OPC_URL = 'opc.tcp://127.0.0.1:4840/freeopcua/server/'            # the endpoint of the factory server of the agent block

try:
    base = mlflow.pyfunc.load_model('models:/blockage-zrule@production').unwrap_python_model()     # the registered z-value rule
except Exception as exc:
    raise RuntimeError('No registered model found: run notebook 02 first. / Kein registriertes Modell gefunden: Führen Sie zuerst Notebook 02 aus.') from exc
print(f'z-value rule: mean {base.mean:.3f}, sd {base.sd:.4f}, alarm if z > {base.z_limit}')

## Step 1: Design the traffic light on stored data
A single minute alarm (`z > limit`) is noisy: a short excursion of the energy would make the light jump. We smooth the alarms into a **health score** between 0 and 1: the exponentially weighted share of alarms. A **half-life** of 60 minutes means that an alarm counts half as much after one hour. The score is turned into a colour by two thresholds. After a blockage and the repair the score starts again at 0.

First the alarms of printer 0 (machine `M01`) for the whole of 2026, computed with the registered rule (the same features as in the service).

In [ ]:
d0, blockages0 = load_printer(DATA_NEW, 0)                         # printer 0 = machine M01
p0 = d0.loc[d0['State'] == 'printing', 'E']                        # energy per printing minute
z0 = (p0.rolling(120).mean() - base.mean) / base.sd                # z-value of E120
alarm0 = (z0 > base.z_limit).astype(float)[z0.notna()]             # 1 = alarm in this minute
print(f'{len(alarm0)} printing minutes, {int(alarm0.sum())} minute alarms, {len(blockages0)} blockages')

**Task:** Fill in the three gaps: the decay factor per minute, the update of the health score for one new minute, and the two thresholds of the traffic light. Rule of thumb for the thresholds: **green below 0.2, yellow up to 0.5, red above.**

In [ ]:
HALF_LIFE = 60                                   # minutes
decay = 0.5 ** (1 / HALF_LIFE)                   # factor per minute: after HALF_LIFE minutes the weight is one half

def update(health, alarm):
    """New health score from the old one and the alarm (0 or 1) of the new minute."""
    return health * decay + alarm * (1 - decay)

def light(health):
    if health < 0.2: return 'green'
    if health < 0.5: return 'yellow'
    return 'red'

print(update(0.0, 1), light(0.1), light(0.3), light(0.8))

Now look at one day of printer 0: the health score and the colour, and the blockages (dashed lines).

In [ ]:
day = alarm0['2026-01-28':'2026-01-28']
stops = list(blockages0[(blockages0 >= day.index[0]) & (blockages0 < pd.Timestamp('2026-01-29'))])
todo, h, hs = list(stops), 0.0, []
for t, a in day.items():
    while todo and t > todo[0]:                                    # after a blockage and repair the monitor starts again
        h = 0.0; todo.pop(0)
    h = update(h, a); hs.append(h)
hs = pd.Series(hs, index=day.index)
fig, ax = plt.subplots(figsize=(10, 3.2))
ax.scatter(hs.index, hs.values, c=[COLORS[light(v)] for v in hs.values], s=6)
for y in (0.2, 0.5): ax.axhline(y, color='grey', lw=0.8, ls=':')
for t in stops:
    ax.axvline(t, color='black', ls='--'); ax.text(t, 1.02, 'blockage', ha='right', fontsize=9)
ax.set_ylim(0, 1.1); ax.set_ylabel('health score'); ax.set_title('Printer 0 (M01), 28 Jan 2026: only printing minutes are shown')
plt.tight_layout(); plt.show()

**Compare a single excursion with a sustained rise. What does the traffic light do in each case, and what does the half-life change?**

A single excursion lifts the score a little (an alarm minute adds 1 - decay, about 1 %), the light stays green or at most turns yellow, and the score decays again. Only alarms in many consecutive minutes push the score over 0.5: the light is red when the energy has been high for about one to two hours, which is the pattern before a blockage. A short half-life (15 min) reacts faster and earlier, but a few noisy minutes are enough to turn it yellow or red; a long half-life (180 min) is calm, but red comes late or not at all.

## Step 2: Half-life and threshold are decisions
The same alarms, different settings, **whole of 2026 for printer 0**. *Announced* = the light was red at some time in the 6 hours before the blockage; *false red* = red without a blockage within 8 hours. After a blockage the score starts again at 0.

In [ ]:
rows = []
for hl in (15, 60, 180):
    dec = 0.5 ** (1 / hl)
    for red_at in (0.35, 0.5, 0.7):
        r = offline_evaluation(alarm0, blockages0, lambda h, a: h * dec + a * (1 - dec), lambda h: 'green' if h < 0.2 else ('yellow' if h < red_at else 'red'))
        rows.append({'half-life [min]': hl, 'red above': red_at, 'announced': f"{r['announced']} of {r['blockages']}", 'share': round(r['share'], 2),
                     'median lead [h]': round(r['median lead [h]'], 1), 'false red per week': round(r['false red per week'], 2)})
pd.DataFrame(rows)

**Which setting would you choose, and who decides? Use the cost ratio: a check costs about 10 minutes, a blockage about 12 hours.**

The table is a trade-off curve, not a ranking: shorter half-life and lower threshold announce more blockages (up to about 85–90 %) and earlier, but cost more false red (about 1 to 1.2 per week and printer instead of 0.05 to 0.5). With a check of 10 minutes against 12 hours, one false red per week and printer (about 10 minutes) is cheap compared with one more announced blockage (up to 12 hours saved), so a setting like half-life 15 to 60 minutes with red at 0.35 to 0.5 is defensible. But the limit is not a model property: the cost ratio belongs to the process owner, and it changes with the number of printers (six printers mean six times the false reds for the same team) and with the real cost of an unnecessary check. The parameters therefore belong into the configuration, with a named owner.

## Step 3: Start the replay server and the service
Both run as separate processes (as in notebook 03). The replay server starts **paused** (`--hold`) and waits until the monitor has subscribed, so that no minute is lost. The first start takes about 20 seconds (the server reads the data of six printers).

If you run this cell a second time, first run the last cell of the notebook (stop), otherwise the ports are still in use.

In [ ]:
from asyncua import Client
replay = subprocess.Popen([sys.executable, 'replay_server.py', '--data', str(DATA_NEW.resolve()), '--start', START, '--hours', str(HOURS), '--speed', str(SPEED), '--hold'],
                          cwd=STORE, stdout=open(STORE / 'replay.log', 'w'), stderr=subprocess.STDOUT)
service = subprocess.Popen([sys.executable, 'blockage_service.py', '--port', '5005', '--tracking-uri', TRACKING_URI],
                           cwd=STORE, stdout=open(STORE / 'service.log', 'w'), stderr=subprocess.STDOUT)

async def opc_ready():
    try:
        async with Client(url=OPC_URL, timeout=2):
            return True
    except Exception:
        return False

service_ok = opc_ok = False
for _ in range(120):
    if replay.poll() is not None or service.poll() is not None:        # a process has ended: waiting longer does not help
        break
    if not service_ok:
        try: service_ok = requests.get(f'{SERVICE_URL}/health', timeout=2).ok
        except requests.RequestException: pass
    opc_ok = opc_ok or await opc_ready()
    if service_ok and opc_ok: break
    await asyncio.sleep(1)
if not (service_ok and opc_ok):
    raise RuntimeError('Replay server or service did not start (service: %s, OPC UA: %s). Most common reason: notebook 02 has not been run, or a port is still in use. '
                       '/ Replay-Server oder Service sind nicht gestartet. Häufigste Ursache: Notebook 02 wurde nicht ausgeführt, oder ein Port ist noch belegt.\n\n--- replay.log ---\n%s\n--- service.log ---\n%s'
                       % (service_ok, opc_ok, (STORE / 'replay.log').read_text()[-800:], (STORE / 'service.log').read_text()[-800:]))
print('replay server and service are running')

Look at what the server offers. The agents of the next block use exactly these nodes. **Task:** fill in the name of the node that holds the energy of the last minute.

In [ ]:
async with Client(url=OPC_URL) as client:
    ns = await client.get_namespace_index('http://ostfalia.de/ipt/factory')
    factory = await client.nodes.objects.get_child([f'{ns}:Factory'])
    print('Factory:', [(await c.read_browse_name()).Name for c in await factory.get_children()])
    m01 = await factory.get_child([f'{ns}:Machines', f'{ns}:M01'])
    print('M01:', [(await c.read_browse_name()).Name for c in await m01.get_children()])
    energy = await m01.get_child([f'{ns}:Energy'])
    print('Energy of the last minute (Wh):', await energy.read_value())

## Step 4: Run the monitor
`prime` fills the window of each printer with the last 360 printing minutes **before** the start from the data archive, so that the monitor can judge from the first minute (a monitor that starts cold needs six printing hours). `start` subscribes to `Energy` and `State` of every machine and releases the replay. The monitor takes **your** `update` and `light`.

Every simulated minute of every printing machine becomes one call of the service; the light is updated, and when it turns red the monitor writes `RepairNeeded = True` and looks up the orders that run on this machine. The replay of 12 hours takes about 75 seconds. **Watch for:** which light turns red first, and does a printer stop while its light is red? You can change the speed while it runs: `await monitor.set_speed(60)` in another cell.

In [ ]:
monitor = await Monitor(OPC_URL, SERVICE_URL, update, light).prime(DATA_NEW, START).start()
view = display(HTML(monitor.html()), display_id=True)
for _ in range(int(HOURS * 60 / SPEED * 60) + 8):                  # real seconds for the replay, plus a few
    await asyncio.sleep(1)
    view.update(HTML(monitor.html()))
for t, name, text in monitor.events:
    print(f'{t:%H:%M}  {name}  {text}')

## Step 5: Did the light announce the blockages?
The monitor kept a log of every minute. `announced` looks at every stop of a printer (state `error`) and checks whether the light was red in the 6 hours before it (after the last repair). `scored = False` means that the monitor could not judge in this window (it was still collecting its first 360 printing minutes). **Task:** count the blockages that were announced and those that could be judged at all.

In [ ]:
log = monitor.dataframe()
table = announced(log)
n_announced = int(table['announced'].sum())
n_scored = int(table['scored'].sum())
print(f'{n_announced} of {len(table)} stops announced by a red light; {n_scored} of {len(table)} could be judged at all')
table

In [ ]:
fig, axes = plt.subplots(len(monitor.machines), 1, figsize=(10, 1.5 * len(monitor.machines)), sharex=True)
for ax, (name, g) in zip(axes, log.groupby('machine')):
    ax.scatter(g['time'], g['health'], c=[COLORS[c] for c in g['light']], s=4)
    ax.set_ylim(0, 1.05); ax.set_ylabel(name, rotation=0, labelpad=22)
    for t in g.loc[(g['state'] == 'error') & (g['state'].shift() != 'error'), 'time']:
        ax.axvline(t, color='black', ls='--')
axes[0].set_title('Health score and light per machine (grey: warm-up, black: stopped, dashed: stop of the printer)')
plt.tight_layout(); plt.show()

**Which stops were not announced, and why? What would you change in the design or in the operation to reach them, and what would it cost?**

The second stop of M01 (20:27) was not announced because the monitor was blind: after the repair at 17:01 the window was reset, and the monitor needs 360 printing minutes again, so it could not judge (scored = False). That is a design decision with a price: keeping the old window would be wrong (it still holds the high energy from before the blockage), so after maintenance the monitor needs about six printing hours. Possible improvements: a shorter window or a second, cheaper rule for the first hours after a repair; both need their own validation. M03 was announced only 0.8 h before the stop: a short lead time may still be enough to finish the job, but not to reschedule it. A stop can also have no precursor in the energy at all; no threshold finds what is not in the signal.

## Step 6: What the monitor hands over
The result for the agent block is not the light but the nodes: `RepairNeeded` per machine and the orders concerned. Read the nodes now (the replay has ended; the monitor resets `RepairNeeded` when a printer is back in operation, so a machine that is still stopped keeps its state).

In [ ]:
print({name: await s['job'].read_value() for name, s in monitor.machines.items()})
print()
for t, name, text in monitor.events:
    if text.startswith('RED'):
        print(f'{t:%H:%M}  {name}  {text}')

**The agents of the next block receive `RepairNeeded` and the orders concerned. What else must they know before they reschedule an order, and who decides?**

A red light is a probability of a blockage, not a fact: about one false red per week and printer (with the shorter half-life) means that an agent which reschedules on every red would reschedule unnecessarily every week. The agent needs the cost ratio (check against blockage), the due times and priorities of the orders, the capacity of the other printers, and the lead time of this alarm (a red light 0.8 h before the stop leaves little room). The agent should propose and name its reason (health score, version of the model, orders concerned); the technician or the production planner decides, until the false-alarm rate and the effect of rescheduling have been measured in operation.

Stop everything when you are done. **The agent block starts its own factory server on port 4840, so keep this port free.**

In [ ]:
await monitor.stop()
replay.terminate(); service.terminate(); replay.wait(); service.wait()
print('replay server, service and monitor stopped')